# 03 — Revision analyses

Analyses added after the internal review: ablation on ResNet50 and DenseNet121, repeated cross-validation,
age-group analysis with confidence intervals, duplicate audit, computational cost, and the ROI quality audit.

Run after notebooks 01 and 02, which create the image cache (`cache/chunk_*.npz`), the features
(`features/*.npy`), and the prediction cache (`hasil_paper/prediksi/*.npz`). Edit `ROOT` in the first cell.

In [ ]:
# R1 — Setup: mount Drive, paths, labels, split
from google.colab import drive
drive.mount('/content/drive')

import os, glob, shutil, json, gc, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# >>> EDIT if your Drive folder is different <<<
ROOT        = "/content/drive/MyDrive/FOLDER TUGAS TPB KELOMPOK 6"
BASE_PATH   = f"{ROOT}/Dataset X-Ray"
LABEL_CSV   = f"{BASE_PATH}/boneage-training-dataset.csv"
SPLIT_CSV   = f"{BASE_PATH}/splits.csv"
DRIVE_CACHE = f"{ROOT}/cache"
DRIVE_FEAT  = f"{ROOT}/features"
DRIVE_OUT   = f"{ROOT}/hasil_paper"
PRED_DIR    = f"{DRIVE_OUT}/prediksi"
REV_DIR     = f"{DRIVE_OUT}/revisi"
FEAT_DIR    = "/content/features"
for d in [DRIVE_FEAT, DRIVE_OUT, PRED_DIR, REV_DIR, FEAT_DIR]:
    os.makedirs(d, exist_ok=True)

SEED = 42
BACKBONES = ["resnet50", "densenet121", "efficientnet_b0"]
VARIANTS  = ["raw", "enh", "roi", "enh_roi"]

labels = pd.read_csv(LABEL_CSV)
splits = pd.read_csv(SPLIT_CSV)
labels["male"] = (labels["male"].astype(str).str.lower()
                  .map({"true": 1, "false": 0, "1": 1, "0": 0}).fillna(0).astype(int))
df = labels.merge(splits[["id", "split"]], on="id")
df["split"] = df["split"].astype(str).str.lower().replace({"validation": "val", "valid": "val"})
df = df.reset_index(drop=True)
tr, va, te = [(df["split"] == s).values for s in ["train", "val", "test"]]
y = df["boneage"].values.astype(np.float32)
male = df["male"].values.astype(np.float32).reshape(-1, 1)
AGE_BINS, AGE_NAMES = [0, 60, 120, 180, 229], ["0-5 y", "5-10 y", "10-15 y", "15-19 y"]
age_grp = pd.Series(pd.cut(y, bins=AGE_BINS, labels=AGE_NAMES, right=False))
print(f"Total {len(df)} | train {tr.sum()} | val {va.sum()} | test {te.sum()}")

In [ ]:
# R2 — Feature extraction on CPU in FP32 (ResNet50-enh re-extracted; DenseNet121 raw/enh/enh_roi added)
import torch, torch.nn as nn
from torchvision import models as M
from tqdm.auto import tqdm
torch.manual_seed(SEED)
dev = "cpu"
WEIGHTS = {"resnet50": M.ResNet50_Weights.IMAGENET1K_V2,
           "densenet121": M.DenseNet121_Weights.IMAGENET1K_V1,
           "efficientnet_b0": M.EfficientNet_B0_Weights.IMAGENET1K_V1}
def get_backbone(name):
    if name == "resnet50":
        m = M.resnet50(weights=WEIGHTS[name]); m.fc = nn.Identity()
    elif name == "densenet121":
        m = M.densenet121(weights=WEIGHTS[name]); m.classifier = nn.Identity()
    else:
        m = M.efficientnet_b0(weights=WEIGHTS[name]); m.classifier = nn.Identity()
    return m.to(dev).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
fname = lambda b, v: f"{b}_{v}.npy"
CHUNKS = sorted(glob.glob(f"{DRIVE_CACHE}/chunk_*.npz"))
def load_variant(v): return np.concatenate([np.load(f)[v] for f in CHUNKS])
@torch.no_grad()
def extract(model, X, bs=64, desc=""):
    out = []
    for i in tqdm(range(0, len(X), bs), desc=desc, leave=False):
        xb = torch.from_numpy(X[i:i + bs]).float().div(255)
        xb = (xb.unsqueeze(1).expand(-1, 3, -1, -1) - MEAN) / STD
        out.append(model(xb).numpy())
    return np.concatenate(out).astype(np.float32)

MARK = f"{DRIVE_FEAT}/resnet50_enh.CPU_FP32_DONE"
old, bak = f"{DRIVE_FEAT}/resnet50_enh.npy", f"{DRIVE_FEAT}/resnet50_enh_GPU_FP16_backup.npy"
if os.path.exists(old) and not os.path.exists(bak): shutil.copy(old, bak)
todo = [] if os.path.exists(MARK) else [("resnet50", "enh")]
todo += [("densenet121", v) for v in ["raw", "enh", "enh_roi"] if not os.path.exists(f"{DRIVE_FEAT}/{fname('densenet121', v)}")]
print("To extract:", todo)
for v in sorted(set(v for _, v in todo)):
    Xv = load_variant(v)
    for b in [b for b, vv in todo if vv == v]:
        F = extract(get_backbone(b), Xv, desc=f"{b}/{v}")
        np.save(f"{FEAT_DIR}/{fname(b, v)}", F); shutil.copy(f"{FEAT_DIR}/{fname(b, v)}", f"{DRIVE_FEAT}/{fname(b, v)}")
        if (b, v) == ("resnet50", "enh"):
            open(MARK, "w").write("re-extracted on CPU FP32")
            for f in glob.glob(f"{PRED_DIR}/resnet50__enh__*"): os.remove(f)
    del Xv; gc.collect()
for b, v in [(b, v) for b in ["resnet50", "densenet121"] for v in VARIANTS] + [("efficientnet_b0", "roi")]:
    loc, drv = f"{FEAT_DIR}/{fname(b, v)}", f"{DRIVE_FEAT}/{fname(b, v)}"
    if not os.path.exists(loc) or os.path.getmtime(drv) > os.path.getmtime(loc): shutil.copy(drv, loc)

In [ ]:
# R3 — Helpers (identical to notebook 02 so that cached predictions are reused)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error as MAEf
MU, SD = y[tr].mean(), y[tr].std(); ys_tr = (y[tr] - MU) / SD
_feat = {}
def get_X(b, v, pca=None, use_sex=True):
    k = (b, v, pca, use_sex)
    if k not in _feat:
        F = np.load(f"{FEAT_DIR}/{fname(b, v)}"); Z = StandardScaler().fit(F[tr]).transform(F)
        if pca: Z = PCA(pca, random_state=SEED).fit(Z[tr]).transform(Z)
        if use_sex: Z = np.hstack([Z, male])
        _feat[k] = Z.astype(np.float32)
    return _feat[k]
def clear_mem(): _feat.clear(); gc.collect()
def make_and_fit(reg, p, Xf):
    if reg == "ridge": return Ridge(alpha=p["alpha"]).fit(Xf[tr], ys_tr)
    if reg == "svr":   return SVR(kernel="rbf", C=p["C"], epsilon=p["eps"], cache_size=2000).fit(Xf[tr], ys_tr)
    if reg == "mlp":
        return MLPRegressor(hidden_layer_sizes=tuple(p["hidden"]), alpha=p["alpha"], batch_size=256, learning_rate_init=1e-3,
                            max_iter=300, early_stopping=True, validation_fraction=0.1, n_iter_no_change=15,
                            random_state=SEED).fit(Xf[tr], ys_tr)
def pkey(p):
    s = "-".join(f"{k}{v}" for k, v in sorted(p.items()))
    return s.replace(" ", "").replace(",", "x").replace("(", "").replace(")", "")
def run_config(b, v, reg, p, pca=None, use_sex=True):
    f = f"{PRED_DIR}/{b}__{v}__{reg}__{pkey(p)}__pca{pca}__sex{int(use_sex)}.npz"
    if os.path.exists(f):
        d = np.load(f); return d["pv"], d["pt"], json.loads(str(d["info"]))
    t0 = time.time(); Xf = get_X(b, v, pca, use_sex); m = make_and_fit(reg, p, Xf)
    pv, pt = m.predict(Xf[va]) * SD + MU, m.predict(Xf[te]) * SD + MU
    info = {"fit_sec": round(time.time() - t0, 1)}
    np.savez(f, pv=pv, pt=pt, info=json.dumps(info)); return pv, pt, info
RIDGE_GRID = [{"alpha": a} for a in [1, 10, 100, 1000, 3000, 10000]]
def tune_ridge(b, v):
    best = None
    for p in RIDGE_GRID:
        pv, pt, _ = run_config(b, v, "ridge", p); mv = MAEf(y[va], pv)
        if best is None or mv < best[0]: best = (mv, p, pv, pt)
    return {"params": best[1], "pv": best[2], "pt": best[3]}
B, n_te = 2000, int(te.sum())
IDX = np.random.default_rng(SEED).integers(0, n_te, size=(B, n_te))
def ci_mae(pt, mask=None):
    e = np.abs(y[te] - pt)
    if mask is None: bs = e[IDX].mean(1)
    else:
        e = e[mask]; r = np.random.default_rng(SEED).integers(0, len(e), size=(B, len(e))); bs = e[r].mean(1)
    return e.mean(), *np.percentile(bs, [2.5, 97.5])
def ci_diff(p_a, p_b):
    d = np.abs(y[te] - p_a) - np.abs(y[te] - p_b); bs = d[IDX].mean(1)
    lo, hi = np.percentile(bs, [2.5, 97.5]); return d.mean(), lo, hi

In [ ]:
# R4 — Preprocessing ablation on ResNet50 and DenseNet121 (Table I, test-subset part)
abl, rows = {}, []
for b in ["resnet50", "densenet121"]:
    for v in VARIANTS:
        abl[(b, v, "ridge")] = tune_ridge(b, v)
        pv, pt, _ = run_config(b, v, "svr", {"C": 10, "eps": 0.1})
        abl[(b, v, "svr")] = {"params": {"C": 10, "eps": 0.1}, "pv": pv, "pt": pt}
    clear_mem()
for (b, v, reg), r in abl.items():
    m, lo, hi = ci_mae(r["pt"])
    row = {"backbone": b, "regressor": reg, "variant": v, "params": str(r["params"]),
           "val_MAE": MAEf(y[va], r["pv"]), "test_MAE": m, "test_CI_lo": lo, "test_CI_hi": hi}
    if v != "raw":
        d, dlo, dhi = ci_diff(abl[(b, "raw", reg)]["pt"], r["pt"])
        row.update({"delta_vs_raw": d, "delta_CI_lo": dlo, "delta_CI_hi": dhi, "signifikan": not (dlo <= 0 <= dhi)})
    rows.append(row)
tabR4 = pd.DataFrame(rows); tabR4.to_csv(f"{REV_DIR}/R4_ablation_resnet_densenet.csv", index=False)
tabR4.round(2)

In [ ]:
# R5 — Repeated stratified 5-fold CV (x2) on the development data (Table I, cross-validation part)
from sklearn.model_selection import RepeatedStratifiedKFold
dev_idx = np.where(tr | va)[0]
strat = (age_grp.astype(str) + "_" + df["male"].astype(str)).values[dev_idx]
folds = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED).split(dev_idx, strat))
cv = {}
for b in ["resnet50", "densenet121"]:
    for v in VARIANTS:
        alpha = abl[(b, v, "ridge")]["params"]["alpha"]
        F = np.load(f"{FEAT_DIR}/{fname(b, v)}")[dev_idx]; yy, sx = y[dev_idx], male[dev_idx]; maes = []
        for i_tr, i_ho in folds:
            sc = StandardScaler().fit(F[i_tr]); mu, sd = yy[i_tr].mean(), yy[i_tr].std()
            m = Ridge(alpha=alpha).fit(np.hstack([sc.transform(F[i_tr]), sx[i_tr]]), (yy[i_tr] - mu) / sd)
            maes.append(MAEf(yy[i_ho], m.predict(np.hstack([sc.transform(F[i_ho]), sx[i_ho]])) * sd + mu))
        cv[(b, v)] = np.array(maes); del F; gc.collect()
rows = []
for (b, v), m in cv.items():
    row = {"backbone": b, "variant": v, "CV_MAE_mean": m.mean(), "CV_MAE_sd": m.std(ddof=1)}
    if v != "raw":
        d = cv[(b, "raw")] - m
        row.update({"delta_mean": d.mean(), "delta_sd": d.std(ddof=1), "delta_min": d.min(), "delta_max": d.max(),
                    "fold_lebih_baik": f"{(d > 0).sum()}/10"})
    rows.append(row)
tabR5 = pd.DataFrame(rows); tabR5.to_csv(f"{REV_DIR}/R5_cv_stabilitas.csv", index=False)
tabR5.round(2)

In [ ]:
# R6 — Age distribution per subset, per-group CI, balanced MAE (Table III)
dist = pd.crosstab(df["split"], age_grp)
dist["mean_age"] = df.groupby("split")["boneage"].mean(); dist["pct_male"] = df.groupby("split")["male"].mean() * 100
dist = dist.loc[["train", "val", "test"]]; dist.to_csv(f"{REV_DIR}/R6_distribusi_usia.csv")
MEMBERS = [("resnet50", "mlp", {"hidden": (512, 256), "alpha": 0.01}),
           ("densenet121", "ridge", {"alpha": 100}),
           ("efficientnet_b0", "mlp", {"hidden": (512, 256), "alpha": 0.01})]
W_ENS = np.array([0.2, 0.5, 0.3])
P = [run_config(b, "roi", reg, p) for b, reg, p in MEMBERS]
pt_ens, pt_single = np.stack([p[1] for p in P], 1) @ W_ENS, P[1][1]
g_te = np.asarray(age_grp)[te]; rows = []
for name, pt in [("ensemble", pt_ens), ("single_densenet_ridge", pt_single)]:
    e = np.abs(y[te] - pt)
    for g in AGE_NAMES:
        mk = g_te == g; m, lo, hi = ci_mae(pt, mk)
        rows.append({"model": name, "group": g, "n": int(mk.sum()), "MAE": m, "CI_lo": lo, "CI_hi": hi,
                     "bias": float((pt[mk] - y[te][mk]).mean())})
    bs = [np.mean([e[r][g_te[r] == g].mean() for g in AGE_NAMES]) for r in IDX[:1000]]
    rows.append({"model": name, "group": "BALANCED", "n": n_te, "MAE": np.mean([e[g_te == g].mean() for g in AGE_NAMES]),
                 "CI_lo": np.percentile(bs, 2.5), "CI_hi": np.percentile(bs, 97.5), "bias": np.nan})
tabR6 = pd.DataFrame(rows); tabR6.to_csv(f"{REV_DIR}/R6_per_usia_CI.csv", index=False)
tabR6.round(2)

In [ ]:
# R7 — Duplicate audit: nearest neighbour in DenseNet121 feature space, confirmed by pixel correlation
F = np.load(f"{FEAT_DIR}/densenet121_raw.npy").astype(np.float32); F /= np.linalg.norm(F, axis=1, keepdims=True)
n = len(F); nn_idx, nn_sim = np.zeros(n, int), np.zeros(n, np.float32)
for i in range(0, n, 1000):
    S = F[i:i + 1000] @ F.T; S[np.arange(S.shape[0]), np.arange(i, i + S.shape[0])] = -1
    nn_idx[i:i + 1000], nn_sim[i:i + 1000] = S.argmax(1), S.max(1)
del F; gc.collect()
cand = pd.DataFrame(sorted({(min(a, b), max(a, b)) for a, b in zip(range(n), nn_idx) if nn_sim[a] >= 0.95}), columns=["i", "j"])
RAW = np.concatenate([np.load(f)["raw"] for f in CHUNKS])
cand["pix_corr"] = [float(np.corrcoef(RAW[a].ravel(), RAW[b].ravel())[0, 1]) for a, b in zip(cand.i, cand.j)]
cand["label_sama"] = (y[cand.i] == y[cand.j]) & (df["male"].values[cand.i] == df["male"].values[cand.j])
cand["beda_split"] = df["split"].values[cand.i] != df["split"].values[cand.j]
cand.to_csv(f"{REV_DIR}/R7b_kandidat_duplikat.csv", index=False); del RAW; gc.collect()
for t in [0.90, 0.95, 0.98, 0.99]:
    s = cand[cand.pix_corr >= t]
    print(f"pixel corr >= {t}: {len(s)} pairs | same label {int(s.label_sama.sum())} | across splits {int(s.beda_split.sum())}")

In [ ]:
# R8 — Computational cost (CPU)
import psutil
Xs = np.load(CHUNKS[0])["roi"][:256]; rows = []
for b in BACKBONES:
    model = get_backbone(b); n_par = sum(p.numel() for p in model.parameters())
    extract(model, Xs[:32], bs=32); t0 = time.time(); extract(model, Xs, bs=32); dt = time.time() - t0
    rows.append({"backbone": b, "params_M": n_par / 1e6, "size_MB_fp32": n_par * 4 / 1e6,
                 "extract_ms_per_img": dt / len(Xs) * 1000, "extract_all_12611_min": dt / len(Xs) * len(df) / 60})
    del model; gc.collect()
tab_cost = pd.DataFrame(rows); reg_rows = []
for (b, reg, p), (pv, pt, info) in zip(MEMBERS, P):
    Xf = get_X(b, "roi")
    m = Ridge(alpha=p["alpha"]).fit(Xf[tr], ys_tr) if reg == "ridge" else \
        MLPRegressor(hidden_layer_sizes=p["hidden"], max_iter=1, random_state=SEED).fit(Xf[tr], ys_tr)
    t0 = time.time(); m.predict(Xf[te]); reg_rows.append({"member": f"{b}-{reg}", "fit_sec": info["fit_sec"],
                                                         "infer_ms_per_img": (time.time() - t0) / n_te * 1000})
clear_mem(); tab_reg = pd.DataFrame(reg_rows)
tab_cost.to_csv(f"{REV_DIR}/R8_biaya_backbone.csv", index=False); tab_reg.to_csv(f"{REV_DIR}/R8_biaya_regressor.csv", index=False)
print(tab_cost.round(2)); print(tab_reg.round(3))

## ROI quality audit
Run the sample cell once, then the audit cell repeatedly: type 20 digits (one per crop, 1 = acceptable,
2 = fingertip truncation, 3 = retained plate/large background, 4 = wrong object, 5 = other) into `JAWABAN`.
If several problems occur, choose the first applicable in the order 4, 2, 3, 5. Compare with the original images
(audit-original cell) before rating fingertip truncation.

In [ ]:
# R9a — Draw 200 random images from the development data (train + val)
N_AUDIT = 200
AUDIT_CSV, SAMPLE_NPZ = f"{REV_DIR}/R9_audit_roi.csv", f"{REV_DIR}/R9_sampel_audit.npz"
CATS = ["Baik", "Ujung jari terpotong", "Pelat/latar besar ikut", "Crop salah objek", "Lainnya"]
if os.path.exists(SAMPLE_NPZ):
    S = np.load(SAMPLE_NPZ); s_idx, s_raw, s_roi, s_fb = S["idx"], S["raw"], S["roi"], S["fb"]
else:
    s_idx = np.sort(np.random.default_rng(SEED).choice(np.where(tr | va)[0], N_AUDIT, replace=False))
    s_raw, s_roi, s_fb = [], [], []
    for c in sorted(set(s_idx // 1000)):
        d = np.load(CHUNKS[c]); sel = s_idx[s_idx // 1000 == c] % 1000
        s_raw.append(d["raw"][sel]); s_roi.append(d["roi"][sel]); s_fb.append(d["fb"][sel])
    s_raw, s_roi, s_fb = map(np.concatenate, (s_raw, s_roi, s_fb))
    np.savez(SAMPLE_NPZ, idx=s_idx, raw=s_raw, roi=s_roi, fb=s_fb)

In [ ]:
# R9b — Audit page by page (edit JAWABAN, then re-run this cell)
JAWABAN = ""
ULANGI_HALAMAN = None
PER = 20
pages = [list(range(p, min(p + PER, N_AUDIT))) for p in range(0, N_AUDIT, PER)]
done = pd.read_csv(AUDIT_CSV).set_index("k")["kategori"].to_dict() if os.path.exists(AUDIT_CSV) else {}
def halaman_aktif():
    if ULANGI_HALAMAN: return ULANGI_HALAMAN - 1
    return next((i for i, ks in enumerate(pages) if not all(k in done for k in ks)), len(pages))
pg, jwb = halaman_aktif(), JAWABAN.replace(" ", "")
if jwb and pg < len(pages):
    ks = pages[pg]
    if len(jwb) == len(ks) and all(c in "12345" for c in jwb):
        for k, c in zip(ks, jwb): done[k] = CATS[int(c) - 1]
        pd.DataFrame({"k": list(done), "kategori": list(done.values()), "id": [df["id"][s_idx[i]] for i in done],
                      "fallback": [int(s_fb[i]) for i in done]}).to_csv(AUDIT_CSV, index=False)
        print(f"Page {pg + 1} saved."); pg = halaman_aktif() if not ULANGI_HALAMAN else pg
    else:
        print(f"JAWABAN must contain exactly {len(ks)} digits 1-5.")
if pg >= len(pages):
    print(f"DONE: {len(done)}/{N_AUDIT}")
else:
    ks = pages[pg]; fig, ax = plt.subplots(4, 5, figsize=(12, 10.5))
    for a, k in zip(ax.flat, ks):
        a.imshow(s_roi[k], cmap="gray"); a.axis("off"); a.set_title(f"No. {ks.index(k) + 1}" + (" [FALLBACK]" if s_fb[k] else ""))
    plt.tight_layout(); plt.show(); plt.close(fig)
    print(f"Page {pg + 1}/{len(pages)} | rated {len(done)}/{N_AUDIT}")

In [ ]:
# R9c — Show the ORIGINAL images of the active page (same numbering)
pg_lihat = (ULANGI_HALAMAN - 1) if ULANGI_HALAMAN else next((i for i, ks in enumerate(pages) if not all(k in done for k in ks)), len(pages))
if pg_lihat < len(pages):
    ks = pages[pg_lihat]; fig, ax = plt.subplots(4, 5, figsize=(12, 10.5))
    for a, k in zip(ax.flat, ks):
        a.imshow(s_raw[k], cmap="gray"); a.axis("off"); a.set_title(f"ORIGINAL No. {ks.index(k) + 1}", color="blue")
    plt.tight_layout(); plt.show(); plt.close(fig)

In [ ]:
# R10 — Audit summary with Wilson 95% CI
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z**2 / n; c = (p + z**2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return 100 * (c - h), 100 * (c + h)
au = pd.read_csv(AUDIT_CSV); n = len(au)
tabR9 = pd.DataFrame([{"kategori": c, "jumlah": int((au.kategori == c).sum()), "persen": 100 * (au.kategori == c).mean(),
                       "CI95": "%.1f-%.1f" % wilson(int((au.kategori == c).sum()), n)} for c in CATS])
tabR9.to_csv(f"{REV_DIR}/R9_ringkasan_audit.csv", index=False)
tabR9.round(1)